# 11_01 Masked language modelling: what does BERT guess when you hide a word?

BERT learned English from text nobody labelled, by hiding words and guessing them back. In this notebook
you play that game with **bert-mini**, a 4-layer BERT with about 11 million weights, and then look at the
other thing pretraining gave it: a vector for every word that depends on the sentence around it.

**How this notebook works.** Every notebook in this course has the same rhythm:

1. **Recall.** Answer from memory before you look anything up. `ask()` tells you at once whether you were right.
2. **Predict, then run.** Before a cell with a surprise in it, write your prediction into `guess()`. The next cell runs the code and `reveal()` compares.
3. **Worked example, then your turn.** One example is done in full; the next, near-identical one has lines marked `# YOUR CODE HERE`.
4. **Check.** A `check_...()` cell tests what you saved, exactly as the checkpoint will, and says what to fix.

Run cells in order with **Shift+Enter**. If you get lost, **Kernel, Restart Kernel and Run All Cells** starts clean.

Running this in Google Colab? This cell sets it up; in CourseLabs it does nothing.

In [ ]:
# Colab setup. In a CourseLabs session this cell does nothing.
import os, sys
if "google.colab" in sys.modules:
    import importlib, importlib.util, subprocess
    LAB, REPO = "lab-nlp-11-bert-and-what-came-after", "/content/nlp-course"
    if not os.path.isdir(REPO):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/fenago/nlp-course.git", REPO], check=True)
    os.chdir(f"{REPO}/{LAB}")
    if not os.path.exists("data"):
        os.symlink("../data", "data")
    os.makedirs("out", exist_ok=True)
    os.environ["NLPLAB_DATA"] = f"{REPO}/data"
    sys.path.insert(0, os.getcwd())
    PIP = {'transformers': 'transformers',
           'torch': 'torch',
           'sklearn': 'scikit-learn',
           'pandas': 'pandas',
           'numpy': 'numpy'}
    missing = [spec for mod, spec in PIP.items() if importlib.util.find_spec(mod) is None]
    if missing:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
        importlib.invalidate_caches()
    print(f"Ready: {LAB} and its data are in {os.getcwd()}; installed {len(missing)} package(s).")
elif not os.path.isdir("/opt/nlplab/data") and os.path.isdir("data"):
    # A downloaded copy on your own computer: the helpers read data/ from here.
    os.environ["NLPLAB_DATA"] = os.path.abspath("data")

In [ ]:
import json
import os
import bertlab
from nlpcheck import ask, guess, reveal, check_11_01

os.makedirs("out", exist_ok=True)
results = {"fills": {}}
print("bert-mini:", bertlab.MINI)

## 1. Recall

**r1.** In Lab 05, how many vectors did word2vec learn for the word "charge"?
(a) one per sentence it appears in, (b) one, whatever the sentence, (c) one per sense, found automatically

**r2.** Where do the labels come from in masked language modelling?
(a) people mark each word's meaning, (b) a dictionary, (c) the text itself: the hidden word is the label

In [ ]:
ask("r1", "")
ask("r2", "")

## 2. The book's sentence

The chapter explains masking with "I love to learn data science", with `data` hidden. `fill_mask()` runs
bert-mini's masked-language-model head and returns its five likeliest WordPieces for the `[MASK]`, with
their probabilities. What will its top guess be? Write one word.

In [ ]:
guess("book_fill", None)   # one word, for example "data"

In [ ]:
s = "I love to learn [MASK] science."
fills = bertlab.fill_mask(s)
results["fills"][s] = fills
for word, p in fills:
    print(f"{word:12s} {p:.3f}")
reveal("book_fill", fills[0][0])

`about`, at nearly half the probability, and `data` is nowhere in the top five. That is not a failure of
the method. "I love to learn about science" is a perfectly good sentence, and nothing in the words around
the gap says *data*: the book's example only works for a reader who already knows the phrase. The model
is answering the question it was trained on, which is "what word fits here", and it fits.

## 3. A Kittiwake sentence, with and without the words to the right

Now a sentence from Kittiwake's price notice, with `month` hidden. First the left context only, then the
same gap with a few more words after it. Predict: does adding words **after** the gap raise the probability
of `month`? (yes or no)

In [ ]:
guess("right_context_matters", None)   # "yes" or "no"

In [ ]:
left = "The Flex 30 plan rises from $30.00 to $32.50 a [MASK]."
both = "The Flex 30 plan rises from $30.00 to $32.50 a [MASK], billed with your usual monthly invoice."
for s in (left, both):
    fills = bertlab.fill_mask(s)
    results["fills"][s] = fills
    print(s)
    print("   ", ", ".join(f"{w} {p:.3f}" for w, p in fills))
month = {s: dict(bertlab.fill_mask(s, k=20)).get("month", 0) for s in (left, both)}
print("probability of month:", month[left], "then", month[both])
reveal("right_context_matters", "yes" if month[both] > month[left] else "no")

With only the left side, bert-mini ranks `day` first and `month` fifth: prices rise "a day", "a year",
"a week" in the text it read. The words *after* the gap, "billed with your usual monthly invoice", nearly
double the probability of `month`, from 0.058 to 0.104. It is still not first, because this is a very small
model, but the guess at the gap changed because of words that come after it. A left-to-right model, which
is what GPT is, could not have used them at that position, because it has not read them yet. That is
what **bidirectional** buys, and it is why the book says BERT reads a token's left and right context in
every layer.

Pretraining did this with 15 percent of the tokens of every sentence in Wikipedia and a large collection
of books. Of the tokens picked, 80 percent became `[MASK]`, 10 percent became a random word and 10 percent
stayed as they were, so the model could never assume that only `[MASK]` positions matter.

## 4. One word, two meanings

Masking is how BERT learned. What it learned shows up in its **contextual vectors**: the vector it gives a
word is computed from the whole sentence. `word_vector(sentence, word)` returns bert-mini's last-layer
vector for a word; `cosine()` compares two, 1 meaning the same direction. Here is "charge" in a billing
sentence and in a battery sentence.

In [ ]:
CHARGE = ["there is a charge on my bill i do not recognise.",
          "they added a late payment charge to my invoice.",
          "my phone will not charge overnight any more.",
          "the handset takes hours to charge with the new cable."]
vecs = [bertlab.word_vector(s, "charge") for s in CHARGE]
print(len(vecs[0]), "numbers per vector")
for i, s in enumerate(CHARGE):
    print(i, s)
print("bill and battery (0 and 2):", bertlab.cosine(vecs[0], vecs[2]))

## 5. Your turn

Is the billing "charge" of sentence 0 closer to the billing "charge" of sentence 1 than to the battery
"charge" of sentence 2? Predict first (yes or no). Then fill in the two lines: `same_sense` is the cosine
between `vecs[0]` and `vecs[1]`, and `different_sense` between `vecs[0]` and `vecs[2]`.

In [ ]:
guess("charge_same_closer", None)   # "yes" or "no"

In [ ]:
same_sense = None        # YOUR CODE HERE: the cosine of vecs[0] and vecs[1]
different_sense = None   # YOUR CODE HERE: the cosine of vecs[0] and vecs[2]

print("same sense:", same_sense, "| different sense:", different_sense)
if same_sense is not None and different_sense is not None:
    reveal("charge_same_closer", "yes" if same_sense > different_sense else "no")
results["same_sense"], results["different_sense"] = same_sense, different_sense
with open("out/11_01_results.json", "w") as f:
    json.dump(results, f, indent=1)
check_11_01()

Word2vec would have printed 1.0 for both: one word, one vector. bert-mini gives "charge" a different vector
in every sentence, closer for the same sense than for the other. The gap is not huge, because this is a very
small model and the sentences share little else, but it is there without anyone ever telling the model that
"charge" has two meanings.

## 6. Exit ticket

In one or two sentences, in the cell below: why could a model that reads only left to right not have
raised the probability of `month` in section 3?

*Your answer here.*

**x1.** How does BERT read its input? (a) left to right, (b) right to left, (c) bidirectionally

**x2.** How many encoder layers does BERT Base stack? (a) 12, (b) 24, (c) 36

In [ ]:
ask("x1", "")
ask("x2", "")